# Experiment 6 - Image Thresholding with Otsu's Method

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

In [ ]:
img = data.coins()

def otsu(src):
    hist = np.bincount(src.ravel(), minlength=256).astype(np.float64)
    p = hist / hist.sum()
    levels = np.arange(256)
    w0 = np.cumsum(p)
    w1 = 1 - w0
    mu_t = (p * levels).sum()
    cum_mu = np.cumsum(p * levels)
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (mu_t * w0 - cum_mu) ** 2 / (w0 * w1)
    between = np.nan_to_num(between)
    return int(np.argmax(between)), between

t_mine, between = otsu(img)
t_cv, mask_cv = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
print("own otsu threshold   :", t_mine)
print("opencv otsu threshold:", int(t_cv))

In [ ]:
mask = np.where(img > t_mine, 255, 0).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].imshow(img, cmap="gray")
axes[0].set_title("coins")
axes[1].hist(img.ravel(), bins=256, color="gray")
axes[1].axvline(t_mine, color="r", ls="--", label=f"t = {t_mine}")
axes[1].plot(between / between.max() * axes[1].get_ylim()[1], color="tab:blue", label="between-class var (scaled)")
axes[1].legend()
axes[1].set_title("histogram")
axes[2].imshow(mask, cmap="gray")
axes[2].set_title("otsu mask")
axes[0].axis("off")
axes[2].axis("off")
plt.show()

print("pixels that differ from opencv:", int((mask != mask_cv).sum()))

In [ ]:
fixed = [60, 100, 160]
fig, axes = plt.subplots(1, len(fixed) + 1, figsize=(18, 4))
for ax, t in zip(axes, fixed):
    ax.imshow(img > t, cmap="gray")
    ax.set_title(f"fixed t = {t}")
ax = axes[-1]
ax.imshow(mask, cmap="gray")
ax.set_title(f"otsu t = {t_mine}")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
blur = cv2.GaussianBlur(img, (5, 5), 0)
t_blur, _ = otsu(blur)
mask_blur = (blur > t_blur).astype(np.uint8) * 255
n, _ = cv2.connectedComponents(mask_blur)

adaptive = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 41, -5)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(mask_blur, cmap="gray")
axes[0].set_title(f"blur + otsu (t={t_blur}), {n - 1} blobs")
axes[1].imshow(adaptive, cmap="gray")
axes[1].set_title("adaptive gaussian, block 41")
for ax in axes:
    ax.axis("off")
plt.show()